In [ ]:
import numpy as np
import pandas as pd
import librosa

from tsai.all import RocketClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Load CSV
music_dta_3 = pd.read_csv('/Users/tarush/Desktop/STA 395/ML_Music_Genre-main/Data/music_with_labels_3_genres.csv')

# Build full paths
base = '/Users/tarush/Desktop/STA 395/ML_Music_Genre-main/Data/Music/genres_original'
music_dta_3["Audio_pth"] = base + "/" + music_dta_3["Label"] + "/" + music_dta_3["Audio"]

# Load waveform only (y) and crop
def load_crop(pth, start=270000, end=390000):
    y, sr = librosa.load(pth, sr=22050)   # fix sr for consistency
    y = y[start:end]
    return y

music_dta_3["x"] = music_dta_3["Audio_pth"].apply(load_crop)

# Train/test split (stratified!)
train_df, test_df = train_test_split(
    music_dta_3,
    test_size=0.25,
    random_state=0,
    stratify=music_dta_3["Label"]
)

# Consistent label encoding
le = LabelEncoder()
le.fit(train_df["Label"])
y_train = le.transform(train_df["Label"])
y_test  = le.transform(test_df["Label"])

# Stack into (n, 1, L)
X_train = np.stack(train_df["x"].to_numpy()).astype(np.float32)[:, None, :]
X_test  = np.stack(test_df["x"].to_numpy()).astype(np.float32)[:, None, :]

print("Shapes:", X_train.shape, X_test.shape)

# ROCKET model (start smaller!)
rck_mod = RocketClassifier(num_kernels=10000, random_state=0)

rck_mod.fit(X_train, y_train)
y_pred = rck_mod.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))

Shapes: (225, 1, 120000) (75, 1, 120000)
